# Colab Training Launcher

Run AntiBERTy or ESM-2 contrastive fine-tuning on Google Colab. This notebook intentionally keeps model and training logic in `src/`; it only mounts persistent storage, prepares the repo, launches training, resumes checkpoints, evaluates the best checkpoint, and plots metrics.

Checkpoints are written to Google Drive via `--output_dir`, so interrupted Colab sessions can resume from `checkpoint_latest.pt`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Keep this on persistent storage. Adjust the path if you prefer a different Drive folder.
DRIVE_OUTPUT = "/content/drive/MyDrive/6.8711/checkpoints"

In [ ]:
import os

REPO_URL = "https://github.com/sgabriel17/contrastive_learning_for_antibody-epitope_reimplementation.git"
REPO_DIR = "/content/contrastive_learning_for_antibody-epitope_reimplementation"
BRANCH = "test1"  # Change if you train from a different branch.

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
# Colab provides Linux/CUDA, so bitsandbytes should install normally here.
!pip install -q -r requirements.txt

# Optional: show key package versions for reproducibility.
!python - <<'PY'
import torch
import transformers
import peft
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('peft', peft.__version__)
print('cuda available', torch.cuda.is_available())
PY

In [ ]:
import pandas as pd
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_bytes / 1e9:.1f} GB free / {total_bytes / 1e9:.1f} GB total")

df = pd.read_pickle("ablang_model/train/rbd_dataset.pd")
print("Dataset shape:", df.shape)
print("DATASET counts:")
print(df["DATASET"].value_counts())
print("Uppercase 12-bin split counts:")
main_epitopes = ["A", "B", "C", "D1", "D2", "E1", "E2.1", "E2.2", "E3", "F1", "F2", "F3"]
print(df[df["DATASET"].isin(["TRAIN", "VAL", "TEST"]) & df["EPITOPE"].isin(main_epitopes)]["DATASET"].value_counts())

In [ ]:
# Edit these per experiment.
ENCODER = "esm2"  # "esm2" or "antiberty"
RUN_NAME = f"{ENCODER}_run1"
OUTPUT_DIR = f"{DRIVE_OUTPUT}/{RUN_NAME}"

EPOCHS = 400
LR = 1e-5
TEMPERATURE = 0.5
CHECKPOINT_EVERY = 5
EVAL_EVERY = 5
SEED = 7
MAX_LENGTH = 512

# Suggested defaults:
# - AntiBERTy: BATCH_SIZE=256, GRAD_ACCUM=1, EVAL_BATCH_SIZE=256
# - ESM-2 on smaller GPUs: BATCH_SIZE=16, GRAD_ACCUM=16, EVAL_BATCH_SIZE=16
if ENCODER == "esm2":
    BATCH_SIZE = 16
    GRAD_ACCUM = 16
    EVAL_BATCH_SIZE = 16
    LOAD_IN_4BIT = True
else:
    BATCH_SIZE = 256
    GRAD_ACCUM = 1
    EVAL_BATCH_SIZE = 256
    LOAD_IN_4BIT = True

# Leave empty for a fresh run. To resume, set to f"{OUTPUT_DIR}/checkpoint_latest.pt".
RESUME = ""

print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM)
print("Output dir:", OUTPUT_DIR)
print("Resume:", RESUME or "fresh run")

In [ ]:
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
resume_arg = f'--resume "{RESUME}"' if RESUME else ""
quant_arg = "--load_in_4bit" if LOAD_IN_4BIT else "--no_load_in_4bit"

cmd = f'''
python src/train.py \
  --encoder {ENCODER} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --eval_batch_size {EVAL_BATCH_SIZE} \
  --grad_accum_steps {GRAD_ACCUM} \
  --lr {LR} \
  --temperature {TEMPERATURE} \
  --max_length {MAX_LENGTH} \
  --seed {SEED} \
  --checkpoint_every {CHECKPOINT_EVERY} \
  --eval_every {EVAL_EVERY} \
  --output_dir "{OUTPUT_DIR}" \
  {quant_arg} \
  {resume_arg}
'''.strip()

print(cmd)
!{cmd}

In [ ]:
# Use this cell after a Colab disconnect. Re-run setup cells first, then this cell.
RESUME = f"{OUTPUT_DIR}/checkpoint_latest.pt"
print("Resume path:", RESUME)

resume_cmd = f'''
python src/train.py \
  --encoder {ENCODER} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --eval_batch_size {EVAL_BATCH_SIZE} \
  --grad_accum_steps {GRAD_ACCUM} \
  --lr {LR} \
  --temperature {TEMPERATURE} \
  --max_length {MAX_LENGTH} \
  --seed {SEED} \
  --checkpoint_every {CHECKPOINT_EVERY} \
  --eval_every {EVAL_EVERY} \
  --output_dir "{OUTPUT_DIR}" \
  {quant_arg} \
  --resume "{RESUME}"
'''.strip()

print(resume_cmd)
# Uncomment to resume:
# !{resume_cmd}

In [ ]:
best_checkpoint = f"{OUTPUT_DIR}/checkpoint_best.pt"
final_eval_json = f"{OUTPUT_DIR}/final_eval.json"

eval_cmd = f'''
python src/evaluate.py \
  --encoder {ENCODER} \
  --checkpoint "{best_checkpoint}" \
  --batch_size {EVAL_BATCH_SIZE} \
  --max_length {MAX_LENGTH} \
  --output_json "{final_eval_json}" \
  {quant_arg}
'''.strip()

print(eval_cmd)
!{eval_cmd}

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

metrics_path = Path(OUTPUT_DIR) / "metrics.json"
metrics = json.loads(metrics_path.read_text())

train_epochs = [m["epoch"] for m in metrics]
train_loss = [m["train_loss"] for m in metrics]
val_rows = [m for m in metrics if "train_vs_val" in m]
val_epochs = [m["epoch"] for m in val_rows]
val_auroc = [m["train_vs_val"]["weighted_auroc"] for m in val_rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_epochs, train_loss)
ax1.set(title="Train loss", xlabel="Epoch", ylabel="NT-Xent loss")
ax2.plot(val_epochs, val_auroc)
ax2.set(title="Validation weighted AUROC", xlabel="Epoch", ylabel="Train-vs-val AUROC")
plt.tight_layout()
fig_path = Path(OUTPUT_DIR) / "training_curves.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("Saved", fig_path)